In [ ]:
import torch_directmlimport re, sys, osfrom pathlib import Pathfrom dml_fastai_utils import setup_dml, get_local_path

In [ ]:
from fastai.text.all import *

In [ ]:
# Initialize DirectML and apply global patchesdml = setup_dml()local_data_path = get_local_path()

In [ ]:
# DML setup handled by setup_dml()

In [ ]:
# Patches handled by setup_dml()

In [ ]:
# =====================================================# 1. Monkey-patch TensorText to add truncate and show methods# =====================================================@patchdef truncate(self: TensorText, n):    """Truncate TensorText to n tokens"""    return type(self)(self[:n])@patch  def show(self: TensorText, ctx=None, **kwargs):    """Show decoded text"""    # Get the tokenizer from kwargs or use a stored one    tokenizer = kwargs.get('tokenizer', getattr(self, '_tokenizer', None))    if tokenizer is not None:        # Decode using HF tokenizer        text = tokenizer.decode(self.cpu().tolist(), skip_special_tokens=True)    else:        # Fallback: just show token IDs        text = str(self.cpu().tolist())    if ctx is None:        print(text)        return text    return show_title(text, ctx=ctx, **kwargs)# =====================================================# 2. Simple Transform - just tokenize to input_ids# =====================================================class HFTokenize(Transform):    """Transform that tokenizes text files using HuggingFace tokenizer"""    def __init__(self, tokenizer, max_length=512):        self.tokenizer = tokenizer        self.max_length = max_length        # Store tokenizer globally for TensorText.show() to access        TensorText._hf_tokenizer = tokenizer    def encodes(self, path: Path):        """Read file and tokenize, returning TensorText with just input_ids"""        text = path.read_text(encoding='utf-8') if path.exists() else ""        # Tokenize - get tensors directly from tokenizer        encoding = self.tokenizer(            text,            truncation='longest_first',            max_length=self.max_length,            padding='max_length',            add_special_tokens=True,            return_tensors='pt'  # Return PyTorch tensors        )        # Squeeze batch dimension [1, seq_len] -> [seq_len]        tensor_text = TensorText(encoding['input_ids'].squeeze(0))        # Attach tokenizer for later decoding        tensor_text._tokenizer = self.tokenizer        return tensor_text    def decodes(self, o):        """Keep as TensorText for show_batch compatibility"""        # Don't decode to string - keep as TensorText        # The actual text decoding happens in show() method        return o# =====================================================# 2. Callback that computes attention_mask on-the-fly# =====================================================class HFCallback(Callback):    """    Computes attention_mask from padded input_ids and injects into model.    This is the ONLY place where HuggingFace-specific logic lives.    """    def __init__(self, pad_idx):        self.pad_idx = pad_idx    def before_batch(self):        """Compute attention_mask from input_ids before each forward pass"""        # Get input_ids from batch        xb = self.xb[0] if isinstance(self.xb, tuple) and len(self.xb) > 0 else self.xb        # Compute attention_mask: 1 for real tokens, 0 for padding        attention_mask = (xb != self.pad_idx).long()        # Inject into model for this forward pass        self.model._attention_mask = attention_mask# =====================================================# 3. Model wrapper that uses injected attention_mask# =====================================================class HFModelWrapper(Module):    """Wrapper to adapt HF model to FastAI"""    def __init__(self, hf_model, pad_idx):        self.hf_model = hf_model        self.pad_idx = pad_idx        self._attention_mask = None    def forward(self, input_ids):        # Get attention_mask from callback (or compute default)        attention_mask = getattr(self, '_attention_mask', None)        if attention_mask is None:            attention_mask = (input_ids != self.pad_idx).long()        # Clear after use (for next batch)        self._attention_mask = None        # Forward pass with attention mask        outputs = self.hf_model(            input_ids=input_ids,            attention_mask=attention_mask        )        return outputs.logits    def __getitem__(self, idx):        """Make model subscriptable for FastAI's freeze_to"""        if idx == 0:            # Return the transformer backbone (everything except classifier)            return self.hf_model.base_model        else:            # Return the classifier head            return self.hf_model.classifier

In [ ]:
# =====================================================# 5. Setup DataLoaders and Model# =====================================================os.makedirs("models", exist_ok=True)model_cache_dir = Path("models")hf_model = "haisongzhang/roberta-tiny-cased"# Load HF model and tokenizerhf_model_obj = AutoModelForSequenceClassification.from_pretrained(    hf_model, num_labels=2, cache_dir=model_cache_dir,    use_safetensors=True)tokenizer = AutoTokenizer.from_pretrained(hf_model)""

In [ ]:
# Load datapath = untar_data(URLs.IMDB, data=local_data_path)# Create DataBlock with HFTransformblocks = (    TransformBlock(        type_tfms=HFTokenize(tokenizer, max_length=512),        dls_kwargs={'before_batch': Pad_Chunk(seq_len=512, pad_idx=tokenizer.pad_token_id)}    ),    CategoryBlock)dblock = DataBlock(    blocks=blocks,    get_items=get_text_files,    get_y=parent_label,    splitter=GrandparentSplitter(valid_name='test'))

In [ ]:
# Create dataloadersdls = dblock.dataloaders(path, bs=1, device=dml, verbose=True, num_workers=0)

In [ ]:
dls.show_batch(max_n=2)

In [ ]:
# =====================================================# 7. Create Learner with custom callback# =====================================================# Wrap the HF modelmodel_wrapped = HFModelWrapper(hf_model_obj, tokenizer.pad_token_id)model_wrapped.to(dml)# Create learnerlearn = Learner(    dls,     model_wrapped,    loss_func=CrossEntropyLossFlat(),    metrics=accuracy,    cbs=[HFCallback(pad_idx=tokenizer.pad_token_id)]).to_fp16(enabled=False)learn.to(dml)print("" + "="*80)print("LEARNER CREATED SUCCESSFULLY")print("="*80)

In [ ]:
print("" + "="*80)print("VERIFYING BATCH STRUCTURE")print("="*80)try:    batch = dls.one_batch()    xb, yb = batch    print(f"Batch shape: {xb.shape}")    print(f"Batch dtype: {xb.dtype}")    print(f"First item type: {type(xb[0])}")    print(f"Labels shape: {yb.shape}")    print(f"Pad token ID: {tokenizer.pad_token_id}")    print(f"Sample tokens (first 20): {xb[0][:20]}")    # Show padding    pad_count = (xb[0] == tokenizer.pad_token_id).sum().item()    print(f"Padding tokens in first sample: {pad_count}/{len(xb[0])}")    print("✓ Batch structure correct!")except Exception as e:    print(f"✗ Batch verification error: {e}")    import traceback    traceback.print_exc()

In [ ]:
# # =====================================================# # 9. Test forward pass with attention_mask injection# # =====================================================# print("" + "="*80)# print("TESTING FORWARD PASS WITH ATTENTION MASK")# print("="*80)# try:#     with torch.no_grad():#         batch = dls.one_batch()#         learn.xb, learn.yb = batch#         # Find our HFCallback#         hf_callback = None#         for cb in learn.cbs:#             if isinstance(cb, HFCallback):#                 hf_callback = cb#                 break#         print(f"Found HFCallback: {hf_callback is not None}")#         # Show what callback does#         print("Before callback:")#         print(f"  Model has _attention_mask: {hasattr(learn.model, '_attention_mask')}")#         print(f"  Input batch shape: {learn.xb[0].shape}")#         print(f"  Input batch type: {type(learn.xb[0])}")#         # Trigger callback manually#         if hf_callback:#             hf_callback.before_batch()#             print("After callback:")#             print(f"  Model has _attention_mask: {hasattr(learn.model, '_attention_mask')}")#             if hasattr(learn.model, '_attention_mask'):#                 print(f"  Attention mask shape: {learn.model._attention_mask.shape}")#                 print(f"  Sample attention mask (first 30): {learn.model._attention_mask[0][:30]}")#                 print(f"  Num real tokens: {learn.model._attention_mask[0].sum().item()}")#                 print(f"  Num pad tokens: {(learn.model._attention_mask[0] == 0).sum().item()}")#         # Forward pass - pass the whole batch, not just first element#         input_tensor = learn.xb[0] if isinstance(learn.xb, tuple) else learn.xb#         pred = learn.model(input_tensor)#         print("Forward pass results:")#         print(f"  Input shape: {input_tensor.shape}")#         print(f"  Output shape: {pred.shape}")#         print(f"  Sample logits: {pred[0]}")#         print("✓ Forward pass successful!")# except Exception as e:#     print(f"✗ Forward pass error: {e}")#     import traceback#     traceback.print_exc()

In [ ]:
learn.fine_tune(1, base_lr=3e-4)

In [ ]:
test_path = Path("test_reviews/test_1.txt")learn.predict(test_path)